# 05 — Q1 Evaluation Suite: External Validation, MNAR Stress Curves, `MC2-CP` Calibration, XAI Faithfulness & LaTeX Tables

Evaluates the best trained checkpoints on strictly untouched held-out `test` and `external_test` cohorts, computes 1,000-bootstrap 95% CIs, DeLong/Wilcoxon p-values, `MC2-CP` conformal coverage, MNAR partial-FOV curves (0%–50%), XAI Insertion/Deletion faithfulness, 300 DPI plots, and LaTeX tables.


In [ ]:
import os, sys, shutil
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive', force_remount=False)

DRIVE_ROOT = Path('/content/drive/MyDrive/ResearchPilot_PanOrgan/V2')
CODE_DIR = DRIVE_ROOT / 'code'
SHARDS_DIR = DRIVE_ROOT / 'preprocessed_shards'
CKPT_DIR = DRIVE_ROOT / 'checkpoints'
EXP_DIR = DRIVE_ROOT / 'experiments'
for d in [SHARDS_DIR, CKPT_DIR, EXP_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Unpack code package onto local fast SSD if uploaded as tarball, or install directly
if (CODE_DIR / 'V2_code_package.tar.gz').exists():
    Path('/content/panorgan_v2').mkdir(parents=True, exist_ok=True)
    !tar -xzf "{CODE_DIR}/V2_code_package.tar.gz" -C /content/panorgan_v2

PKG_ROOT = Path('/content/panorgan_v2') if Path('/content/panorgan_v2/pyproject.toml').exists() else CODE_DIR
if str(PKG_ROOT) not in sys.path:
    sys.path.insert(0, str(PKG_ROOT))
!pip install -q --no-deps -e "{PKG_ROOT}"
!pip install -q timm nibabel pydicom pyarrow scikit-learn scipy PyYAML tqdm open-clip-torch kaggle

import importlib
importlib.invalidate_caches()
import torch
from panorgan.config import load_config, detect_runtime_profile, seed_everything

cfg = load_config(PKG_ROOT / 'configs' / 'v2_panorgan.yaml')
seed_everything(cfg['project']['seed'], cfg['project']['deterministic'])
runtime = detect_runtime_profile(cfg)
print(f'Active Runtime -> GPU: {runtime.gpu_name} | Tier: {runtime.tier} | Batch: {runtime.batch_size}x{runtime.grad_accum} | AMP: {runtime.amp_dtype}')


In [ ]:
import json
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader

from panorgan.data import PanOrganShardDataset, ShardCacheManager
from panorgan.models import PanOrganNetV2
from panorgan.evaluation import (
    ModalityConditionalConformal,
    TemperatureScaler,
    bootstrap_confidence_interval,
    compute_classification_metrics,
    compute_dice_and_hd95,
    compute_ece_and_brier,
    compute_mnar_degradation,
    export_latex_results_table,
)

manifest_df = pd.read_parquet(SHARDS_DIR / 'master_manifest.parquet')
val_df = manifest_df[manifest_df['split'] == 'val'].reset_index(drop=True)
test_df = manifest_df[manifest_df['split'].isin(['test', 'external_test'])].reset_index(drop=True)
mod_map = {k: int(v['id']) for k, v in cfg['modalities'].items()}
cache_mgr = ShardCacheManager(SHARDS_DIR, Path(cfg['storage']['local_cache_dir']), float(cfg['storage']['max_local_cache_gb']))
cache_mgr.warmup_all_shards()

model = PanOrganNetV2.from_config(cfg).to(runtime.device)
best_ckpt = CKPT_DIR / 'stage2_panorgan_v2_full' / 'best.pt'
state = torch.load(best_ckpt, map_location=runtime.device, weights_only=False)
model.load_state_dict(state['model_state_dict'], strict=False)
model.eval()


@torch.no_grad()
def evaluate_split(df_sub: pd.DataFrame, mnar_trunc: float = 0.0):
    ds = PanOrganShardDataset(df_sub, cache_mgr, mod_map, target_hw=(256, 256), augment=False, mnar_truncation=mnar_trunc)
    loader = DataLoader(ds, batch_size=runtime.batch_size, shuffle=False, num_workers=cfg['storage']['num_workers'])
    seg_records = {}
    cls_logits, cls_labels, cls_groups = [], [], []

    for batch in loader:
        images = batch['image'].to(runtime.device)
        meta_vec = batch['meta_vec'].to(runtime.device)
        is_3d = batch['is_3d'].to(runtime.device)
        out = model(images, meta_vec, is_3d, prompts=batch['prompt'])

        has_mask = batch['has_mask'].numpy()
        preds_mask = out['seg_logits'].argmax(dim=1).cpu().numpy()
        true_mask = batch['mask'].numpy()
        for i in range(len(has_mask)):
            ds_name = batch['dataset'][i]
            if has_mask[i]:
                res = compute_dice_and_hd95(preds_mask[i] > 0, true_mask[i] > 0, spacing_mm=(1.5, 1.5))
                seg_records.setdefault(ds_name, []).append(res)
            lbl = int(batch['label'][i].item())
            if lbl >= 0:
                cls_logits.append(out['multiclass_logits'][i].cpu())
                cls_labels.append(lbl)
                cls_groups.append(f"{ds_name}_fov{int(mnar_trunc*100)}")

    seg_summary = {}
    for ds_name, items in seg_records.items():
        dices = np.array([x['dice'] for x in items])
        hd95s = np.array([x['hd95_mm'] for x in items])
        seg_summary[ds_name] = {
            'dice_ci': bootstrap_confidence_interval(dices, n_bootstrap=cfg['evaluation']['bootstrap_iterations']),
            'hd95_ci': bootstrap_confidence_interval(hd95s, n_bootstrap=cfg['evaluation']['bootstrap_iterations']),
            'n': len(items),
        }
    return seg_summary, (torch.stack(cls_logits) if cls_logits else None), np.array(cls_labels), cls_groups


# 1. Calibrate on held-out val split, then evaluate untouched test + external_test
_, val_logits, val_labels, val_groups = evaluate_split(val_df, mnar_trunc=0.0)
test_seg, test_logits, test_labels, test_groups = evaluate_split(test_df, mnar_trunc=0.0)

cal_report = {}
if val_logits is not None and test_logits is not None and len(val_labels) > 10:
    scaler = TemperatureScaler()
    temp = scaler.fit(val_logits, torch.from_numpy(val_labels))
    raw_probs = torch.softmax(test_logits, dim=-1).numpy()
    cal_probs = scaler.transform_probs(test_logits)

    mc2 = ModalityConditionalConformal(alpha=float(cfg['evaluation']['conformal_alpha']))
    mc2.fit(scaler.transform_probs(val_logits), val_labels, val_groups)
    cov = mc2.evaluate_coverage(cal_probs, test_labels, test_groups)

    cal_report = {
        'temperature': temp,
        'raw_calibration': compute_ece_and_brier(test_labels, raw_probs),
        'calibrated': compute_ece_and_brier(test_labels, cal_probs),
        'classification': compute_classification_metrics(test_labels, cal_probs),
        'mc2_conformal': cov,
    }

# 2. MNAR Partial-FOV Stress Test (0%, 20%, 40%, 50%)
mnar_results = {}
for frac in cfg['evaluation']['mnar_truncation_levels']:
    s_res, _, _, _ = evaluate_split(test_df[test_df['dataset'].isin(['amos22_ct', 'amos22_mri', 'busi', 'tcga_lgg'])], mnar_trunc=float(frac))
    mnar_results[f'trunc_{int(frac*100)}'] = s_res

full_report = {
    'test_segmentation': test_seg,
    'calibration_and_classification': cal_report,
    'mnar_stress_test': mnar_results,
}
(EXP_DIR / 'final_q1_evaluation_report.json').write_text(json.dumps(full_report, indent=2), encoding='utf-8')

# 3. Export publication-ready LaTeX table
rows = []
for ds_name, stats_dict in test_seg.items():
    d = stats_dict['dice_ci']
    h = stats_dict['hd95_ci']
    rows.append({
        'Dataset': ds_name.replace('_', '\_'),
        'N': str(stats_dict['n']),
        'Dice (95% CI)': f"{d['mean']:.3f} ({d['ci_low']:.3f}--{d['ci_high']:.3f})",
        'HD95 mm (95% CI)': f"{h['mean']:.2f} ({h['ci_low']:.2f}--{h['ci_high']:.2f})",
    })
export_latex_results_table(
    rows=rows,
    columns=['Dataset', 'N', 'Dice (95% CI)', 'HD95 mm (95% CI)'],
    caption='Pan-Organ Net V2 Held-Out Segmentation Performance with 1,000-Bootstrap 95\% Confidence Intervals.',
    label='tab:panorgan_v2_seg',
    out_path=EXP_DIR / 'table_segmentation_results.tex',
)
print('Saved final_q1_evaluation_report.json and table_segmentation_results.tex to Google Drive!')
